In [ ]:
!pip -q install nibabel SimpleITK lungmask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 98.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, glob
import numpy as np
import SimpleITK as sitk
from lungmask import mask as lungmask

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/ct"
OUT_DIR  = "/content/drive/MyDrive/Colab Notebooks/masks"
os.makedirs(OUT_DIR, exist_ok=True)

ct_list = sorted(glob.glob(os.path.join(DATA_DIR, "*.nii.gz")))
print("Găsit", len(ct_list), "CT-uri")

Găsit 10 CT-uri


In [ ]:
for ct_path in ct_list:
    ct = sitk.ReadImage(ct_path)
    seg_np = lungmask.apply(ct)
    seg_bin = (seg_np > 0).astype(np.uint8)

    seg_img = sitk.GetImageFromArray(seg_bin)
    seg_img.CopyInformation(ct)

    out_path = os.path.join(
        OUT_DIR,
        os.path.basename(ct_path).replace("_CT.nii.gz", "_segmentation_ct.nrrd")
    )
    sitk.WriteImage(seg_img, out_path, useCompression=True)
    print("Salvat:", out_path)

20it [00:07,  2.61it/s]

lungmask 2025-11-02 10:06:44 Postprocessing



100%|██████████| 93/93 [00:03<00:00, 24.44it/s]


Salvat: /content/drive/MyDrive/Colab Notebooks/masks/10111_segmentation_ct.nrrd


 97%|█████████▋| 18/18.55 [00:07<00:00,  2.56it/s]/usr/local/lib/python3.12/dist-packages/tqdm/std.py:636: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
102%|██████████| 19/18.55 [00:07<00:00,  2.60it/s]

lungmask 2025-11-02 10:07:08 Postprocessing


... (70 outputs left out)

100%|██████████| 15/14.95 [00:06<00:00,  2.45it/s]

lungmask 2025-11-02 10:13:36 Postprocessing



100%|██████████| 33/33 [00:01<00:00, 32.87it/s]


Salvat: /content/drive/MyDrive/Colab Notebooks/masks/9111_segmentation_ct.nrrd


In [ ]:
print(ct_list)

['/content/drive/MyDrive/Colab Notebooks/lungs/10111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/2011_PET.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/22011_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/23111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/24111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/25111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/26011_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/28111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/29111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/30111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/3111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/41111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/4111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/42111_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/lungs/43011_CT.nii.gz', '/content/drive/MyDrive/Colab Notebooks/l

In [ ]:
ct_list_2 = ['/content/drive/MyDrive/Colab Notebooks/lungs/2011_CT.nii.gz']

In [ ]:
for ct_path in ct_list_2:
    ct = sitk.ReadImage(ct_path)
    seg_np = lungmask.apply(ct)
    seg_bin = (seg_np > 0).astype(np.uint8)

    seg_img = sitk.GetImageFromArray(seg_bin)
    seg_img.CopyInformation(ct)

    out_path = os.path.join(
        OUT_DIR,
        os.path.basename(ct_path).replace("_CT.nii.gz", "_segmentation_ct.nrrd")
    )
    sitk.WriteImage(seg_img, out_path, useCompression=True)
    print("Salvat:", out_path)

17it [00:06,  2.67it/s]                           

lungmask 2025-11-02 10:36:02 Postprocessing



100%|██████████| 59/59 [00:02<00:00, 25.42it/s]


Salvat: /content/drive/MyDrive/Colab Notebooks/masks/2011_segmentation_ct.nrrd


In [ ]:
def _read_any(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return sitk.ReadImage(path)

In [ ]:
def make_pet_lung_mask_for_case(
    pet_path: str,
    ct_path: str,
    ct_lung_mask_path: str,
    out_mask_pet_path: str,
    use_float64: bool = False,
    verbose: bool = True,
):
    pet = sitk.ReadImage(pet_path)
    ct  = sitk.ReadImage(ct_path)
    mct = sitk.ReadImage(ct_lung_mask_path)

    fpix = sitk.sitkFloat64 if use_float64 else sitk.sitkFloat32
    petf = sitk.Cast(pet, fpix)
    ctf  = sitk.Cast(ct,  fpix)

    initial = sitk.CenteredTransformInitializer(
        petf, ctf, sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY
    )
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(50)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(0.2, seed=1234)
    reg.SetInterpolator(sitk.sitkLinear)
    reg.SetOptimizerAsRegularStepGradientDescent(
        learningRate=2.0, minStep=1e-4, numberOfIterations=200,
        relaxationFactor=0.5, gradientMagnitudeTolerance=1e-8
    )
    reg.SetOptimizerScalesFromPhysicalShift()
    reg.SetShrinkFactorsPerLevel([4,2,1])
    reg.SetSmoothingSigmasPerLevel([2,1,0])
    reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    reg.SetInitialTransform(initial, inPlace=False)
    T = reg.Execute(petf, ctf)

    mct = sitk.Cast(mct, sitk.sitkUInt8)
    mpet = sitk.Resample(mct, pet, T, sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)

    os.makedirs(os.path.dirname(out_mask_pet_path), exist_ok=True)
    sitk.WriteImage(mpet, out_mask_pet_path, useCompression=True)

    if verbose:
        print(f"[OK] {os.path.basename(pet_path)} → {os.path.basename(out_mask_pet_path)}")


In [ ]:
import re

In [ ]:
def batch_make_pet_lung_masks_convention(
    ct_dir: str,
    pet_dir: str,
    mask_ct_dir: str,
    mask_pet_dir: str,
    use_float64: bool = False,
):
    os.makedirs(mask_pet_dir, exist_ok=True)

    def id_from(pattern_removed: str, fname: str) -> str:
        base = os.path.basename(fname)
        base = re.sub(r"\.nii(\.gz)?$|\.nrrd$", "", base, flags=re.IGNORECASE)
        return base.replace(pattern_removed, "")

    ct_paths  = sorted(glob.glob(os.path.join(ct_dir,  "*_CT.nii.gz")))
    pet_paths = sorted(glob.glob(os.path.join(pet_dir, "*_PET.nii.gz")))
    mct_paths = sorted(glob.glob(os.path.join(mask_ct_dir, "*_segmentation_ct.nrrd")))

    ct_by_id   = {id_from("_CT",  p): p for p in ct_paths}
    pet_by_id  = {id_from("_PET", p): p for p in pet_paths}
    mct_by_id  = {id_from("_segmentation_ct", p): p for p in mct_paths}

    ids = sorted(set(pet_by_id) & set(ct_by_id) & set(mct_by_id))
    missing = sorted((set(pet_by_id) | set(ct_by_id) | set(mct_by_id)) - set(ids))

    for pid in ids:
        pet_path = pet_by_id[pid]
        ct_path  = ct_by_id[pid]
        mct_path = mct_by_id[pid]

        out_mask_pet = os.path.join(mask_pet_dir, f"{pid}_segmentation_pet.nrrd")

        make_pet_lung_mask_for_case(
            pet_path=pet_path,
            ct_path=ct_path,
            ct_lung_mask_path=mct_path,
            out_mask_pet_path=out_mask_pet,
            use_float64=use_float64,
            verbose=True
        )

    if missing:
        print("\n[INFO] ID-uri incomplete (lipsesc CT, PET sau mask_ct):")
        for pid in missing:
            print(" -", pid)

In [ ]:
batch_make_pet_lung_masks_convention(
    ct_dir = "/content/drive/MyDrive/Colab Notebooks/ct",
    pet_dir = "/content/drive/MyDrive/Colab Notebooks/pet",
    mask_ct_dir = "/content/drive/MyDrive/Colab Notebooks/masks_ct",
    mask_pet_dir = "/content/drive/MyDrive/Colab Notebooks/masks_pet",
    use_float64 = False
)

[OK] 10111_PET.nii.gz → 10111_segmentation_pet.nrrd
[OK] 2011_PET.nii.gz → 2011_segmentation_pet.nrrd
[OK] 22011_PET.nii.gz → 22011_segmentation_pet.nrrd
[OK] 23111_PET.nii.gz → 23111_segmentation_pet.nrrd
[OK] 24111_PET.nii.gz → 24111_segmentation_pet.nrrd
[OK] 25111_PET.nii.gz → 25111_segmentation_pet.nrrd
[OK] 26011_PET.nii.gz → 26011_segmentation_pet.nrrd
[OK] 28111_PET.nii.gz → 28111_segmentation_pet.nrrd
[OK] 29111_PET.nii.gz → 29111_segmentation_pet.nrrd
[OK] 30111_PET.nii.gz → 30111_segmentation_pet.nrrd
[OK] 3111_PET.nii.gz → 3111_segmentation_pet.nrrd
[OK] 4111_PET.nii.gz → 4111_segmentation_pet.nrrd
[OK] 41111_PET.nii.gz → 41111_segmentation_pet.nrrd
[OK] 42111_PET.nii.gz → 42111_segmentation_pet.nrrd
[OK] 43011_PET.nii.gz → 43011_segmentation_pet.nrrd
[OK] 5111_PET.nii.gz → 5111_segmentation_pet.nrrd


RuntimeError: Exception thrown in SimpleITK ImageRegistrationMethod_Execute: /tmp/SimpleITK-build/ITK-prefix/include/ITK-5.4/itkSmoothingRecursiveGaussianImageFilter.hxx:214:
ITK ERROR: SmoothingRecursiveGaussianImageFilter(0x3d251a90): The number of pixels along dimension 2 is less than 4. This filter requires a minimum of four pixels along the dimension to be processed.

In [ ]:
def batch_make_pet_lung_masks_convention_validate(
    ct_dir: str,
    pet_dir: str,
    mask_ct_dir: str,
    mask_pet_dir: str,
    use_float64: bool = False,
):
    import os, re, glob

    os.makedirs(mask_pet_dir, exist_ok=True)

    def id_from(pattern_removed: str, fname: str) -> str:
        base = os.path.basename(fname)
        base = re.sub(r"\.nii(\.gz)?$|\.nrrd$", "", base, flags=re.IGNORECASE)
        return base.replace(pattern_removed, "")

    ct_paths  = sorted(glob.glob(os.path.join(ct_dir,  "*_CT.nii.gz")))
    pet_paths = sorted(glob.glob(os.path.join(pet_dir, "*_PET.nii.gz")))
    mct_paths = sorted(glob.glob(os.path.join(mask_ct_dir, "*_segmentation_ct.nrrd")))

    ct_by_id   = {id_from("_CT",  p): p for p in ct_paths}
    pet_by_id  = {id_from("_PET", p): p for p in pet_paths}
    mct_by_id  = {id_from("_segmentation_ct", p): p for p in mct_paths}

    ids = sorted(set(pet_by_id) & set(ct_by_id) & set(mct_by_id))
    missing = sorted((set(pet_by_id) | set(ct_by_id) | set(mct_by_id)) - set(ids))

    for pid in ids:
        pet_path = pet_by_id[pid]
        ct_path  = ct_by_id[pid]
        mct_path = mct_by_id[pid]
        out_mask_pet = os.path.join(mask_pet_dir, f"{pid}_segmentation_pet.nrrd")

        try:
            make_pet_lung_mask_for_case(
                pet_path=pet_path,
                ct_path=ct_path,
                ct_lung_mask_path=mct_path,
                out_mask_pet_path=out_mask_pet,
                use_float64=use_float64,
                verbose=True
            )
        except Exception as e:
            print(f"[EROARE] Eroare la procesarea ID-ului '{pid}': {e}")
            continue

    if missing:
        print("\n[INFO] ID-uri incomplete (lipsesc CT, PET sau mask_ct):")
        for pid in missing:
            print(" -", pid)


In [ ]:
for ct_path in ct_list:
    ct = sitk.ReadImage(ct_path)
    seg_np = lungmask.apply(ct)
    seg_bin = (seg_np > 0).astype(np.uint8)

    seg_img = sitk.GetImageFromArray(seg_bin)
    seg_img.CopyInformation(ct)

    out_path = os.path.join(
        OUT_DIR,
        os.path.basename(ct_path).replace("_CT.nii.gz", "_segmentation_ct.nrrd")
    )
    sitk.WriteImage(seg_img, out_path, useCompression=True)
    print("Salvat:", out_path)

Downloading: "https://github.com/JoHof/lungmask/releases/download/v0.0/unet_r231-d5d2fc3d.pth" to /root/.cache/torch/hub/checkpoints/unet_r231-d5d2fc3d.pth


100%|██████████| 119M/119M [00:00<00:00, 127MB/s]
100%|██████████| 13/13.0 [00:05<00:00,  2.22it/s]

lungmask 2025-11-02 14:37:21 Postprocessing



100%|██████████| 25/25 [00:00<00:00, 41.65it/s]


Salvat: /content/drive/MyDrive/Colab Notebooks/masks/11111_segmentation_ct.nrrd


 96%|█████████▌| 14/14.6 [00:05<00:00,  2.63it/s]/usr/local/lib/python3.12/dist-packages/tqdm/std.py:636: TqdmWarning: clamping frac to range [0, 1]
  full_bar = Bar(frac,
103%|██████████| 15/14.6 [00:05<00:00,  2.67it/s]

... (31 outputs left out)

100%|██████████| 15/14.95 [00:05<00:00,  2.55it/s]

lungmask 2025-11-02 14:40:31 Postprocessing



100%|██████████| 33/33 [00:00<00:00, 34.48it/s]


Salvat: /content/drive/MyDrive/Colab Notebooks/masks/9111_segmentation_ct.nrrd


In [ ]:
batch_make_pet_lung_masks_convention_validate(
    ct_dir = "/content/drive/MyDrive/Colab Notebooks/ct",
    pet_dir = "/content/drive/MyDrive/Colab Notebooks/pet",
    mask_ct_dir = "/content/drive/MyDrive/Colab Notebooks/masks_ct",
    mask_pet_dir = "/content/drive/MyDrive/Colab Notebooks/masks_pet",
    use_float64 = False
)

[OK] 11111_PET.nii.gz → 11111_segmentation_pet.nrrd
[OK] 12111_PET.nii.gz → 12111_segmentation_pet.nrrd
[OK] 13111_PET.nii.gz → 13111_segmentation_pet.nrrd
[OK] 15011_PET.nii.gz → 15011_segmentation_pet.nrrd
[OK] 16111_PET.nii.gz → 16111_segmentation_pet.nrrd
[OK] 40011_PET.nii.gz → 40011_segmentation_pet.nrrd
[OK] 44111_PET.nii.gz → 44111_segmentation_pet.nrrd
[OK] 7011_PET.nii.gz → 7011_segmentation_pet.nrrd
[OK] 8011_PET.nii.gz → 8011_segmentation_pet.nrrd
[OK] 9111_PET.nii.gz → 9111_segmentation_pet.nrrd
